[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与统计热身

**Kernel：Evals Course（`evals`）** · 本 notebook **纯 CPU**，从头到尾约 5–10 分钟。

配套讲解：[00_overview.html](./00_overview.html) · 课程主页：[../index.html](../index.html)

本 notebook 做四件事：

1. **包版本自检** —— 核心统计栈 + LLM 栈，缺失的用红色标出；
2. **device 与 API key 检测** —— cuda / mps / cpu；API key 只报告**是否存在**，绝不打印值；
3. **统计栈冒烟测试** —— 用一次微型"benchmark 测量"跑通 numpy / scipy；
4. **两道 ✏️ 热身练习** —— Wald CI 与 bootstrap CI：`TODO` 骨架 + `assert` 自动判分，这是全课练习的固定模式。

> **使用方法**：从上到下依次运行。练习 cell 完成 `TODO` 后删除 `raise NotImplementedError`，再跑紧随其后的自测 cell——打印 `✅` 即通过；做不出再看文末 📖 参考答案。

### 各模块算力需求一览

| 模块 | 内容 | 算力 |
|---|---|---|
| 00 | 环境自检与统计热身 | CPU |
| 01 | 评测分类学与 benchmark 全景 | CPU |
| 02 | 统计严谨性（误差棒 / bootstrap / 检验） | CPU |
| 03 | 答案抽取与 prompt 敏感性 | CPU + `Qwen/Qwen2.5-1.5B-Instruct`（GPU/MPS 可选） |
| 04 | LLM-as-a-Judge | CPU + 小模型 judge（GPU/MPS 可选；API key 可选） |
| 05 | 污染与饱和 | CPU |
| 06 | 能力引出与 pass@k | CPU + `Qwen/Qwen2.5-1.5B-Instruct`（GPU/MPS 可选） |
| 07 | Eval harness 工程 | CPU |
| 08 | Arena / Elo / time-horizon / 报告 | CPU |

torch / transformers / datasets 现在缺失**不影响模块 00–02**，在进入模块 03 之前装好即可。

In [ ]:
# ── 1) 包版本自检：核心统计栈（必装）+ LLM 栈（模块 03/04/06 需要）──
# 缺失的包用红色标出，并给出对应的 pip install 命令。
import importlib

RED, GREEN, YELLOW, RESET = "\033[91m", "\033[92m", "\033[93m", "\033[0m"

CORE = [("numpy", "numpy"), ("scipy", "scipy"), ("statsmodels", "statsmodels"),
        ("sklearn", "scikit-learn"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]
LLM = [("torch", "torch"), ("transformers", "transformers"), ("datasets", "datasets")]

def check(packages, label):
    print(f"== {label} ==")
    missing = []
    for mod_name, pip_name in packages:
        try:
            mod = importlib.import_module(mod_name)
            ver = getattr(mod, "__version__", "?")
            print(f"  {GREEN}OK{RESET}      {mod_name:<14} {ver}")
        except ImportError:
            missing.append(pip_name)
            print(f"  {RED}MISSING {mod_name:<14} 未安装 -> pip install {pip_name}{RESET}")
    return missing

missing = check(CORE, "核心统计栈（00–08 全程需要）")
missing += check(LLM, "LLM 栈（模块 03/04/06 才需要）")

print()
if missing:
    print(f"{YELLOW}共缺失 {len(missing)} 个包，一次性补齐：{RESET}")
    print("  pip install " + " ".join(missing))
else:
    print(f"{GREEN}✅ 全部依赖就绪{RESET}")

In [ ]:
# ── 2) device 探测 + API key 检测 ─────────────────────────────
# 安全纪律：key 只打印 bool（是否存在），绝不打印值本身。
import os

device = "cpu"
try:
    import torch
    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = "mps"
    print(f"torch 可用，device = {device}")
except ImportError:
    print("torch 未安装：模块 00–02 用不到，进入模块 03 前补装即可")

print()
print("API key（全部可选，缺失时课程 notebook 会自动回退到本地小模型/跳过）：")
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "HF_TOKEN"]:
    print(f"  {key:<18} 存在: {bool(os.environ.get(key))}")

In [ ]:
# ── 3) 统计栈冒烟测试：一次微型"benchmark 测量"────────────────
# 设定：模型真实正确率 p_true=0.7，"考" n=200 道题（Bernoulli 抽样），
# 然后：算观测正确率 p_hat -> 95% Wald CI -> 用 scipy 检验 H0: p=0.5。
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
p_true, n = 0.7, 200
results = rng.binomial(1, p_true, size=n)   # 每题 1=答对 / 0=答错

p_hat = results.mean()
se = np.sqrt(p_hat * (1 - p_hat) / n)
lo, hi = p_hat - 1.96 * se, p_hat + 1.96 * se

print(f"真实正确率 p_true = {p_true}")
print(f"观测正确率 p_hat  = {p_hat:.3f}   <- 注意它偏离真值好几个百分点：这就是抽样噪声")
print(f"95% Wald CI       = [{lo:.3f}, {hi:.3f}]（半宽 {1.96 * se:.3f}）")
print(f"CI 是否盖住真值   = {lo <= p_true <= hi}")

test = stats.binomtest(int(results.sum()), n, p=0.5, alternative="two-sided")
print(f"\nH0: p=0.5 的 binomtest p-value = {test.pvalue:.2e}  -> 远小于 0.05，拒绝 H0")
print("\n✅ 统计栈冒烟测试通过（numpy / scipy 工作正常）")

## ✏️ 练习 1：实现 Wald 置信区间

讲解第 1 节的公式，亲手写一遍。给定观测正确率 $\hat p$、题目数 $n$、临界值 $z$（默认 1.96 对应 95%）：

$$\mathrm{CI} = \Big[\,\hat p - z\sqrt{\tfrac{\hat p(1-\hat p)}{n}},\;\; \hat p + z\sqrt{\tfrac{\hat p(1-\hat p)}{n}}\,\Big]$$

实现 `wald_ci(p_hat, n, z=1.96)`，返回 tuple `(lo, hi)`。

**提示**：3 行内可完成，`np.sqrt(...)` 或 `(...) ** 0.5` 均可。留意自测中 `p_hat=1.0` 的边界用例——Wald 区间在 0/1 边界会**塌缩成零宽度**，这是它的著名缺陷（模块 02 讲 Wilson / Clopper–Pearson 如何修复）。

In [ ]:
import numpy as np

def wald_ci(p_hat, n, z=1.96):
    # TODO: 计算标准误 se = sqrt(p_hat * (1 - p_hat) / n)
    # TODO: 返回 (p_hat - z * se, p_hat + z * se)
    raise NotImplementedError("完成上面两个 TODO 后删除此行")

In [ ]:
# ── 练习 1 自测（全部通过会打印 ✅）──────────────────────────
lo, hi = wald_ci(0.8, 100)
assert abs(lo - 0.7216) < 1e-6 and abs(hi - 0.8784) < 1e-6, \
    f"wald_ci(0.8, 100) 应为 (0.7216, 0.8784)，你返回了 ({lo}, {hi})"

lo, hi = wald_ci(0.5, 25)
assert abs(lo - 0.304) < 1e-6 and abs(hi - 0.696) < 1e-6, \
    f"wald_ci(0.5, 25) 应为 (0.304, 0.696)，你返回了 ({lo}, {hi})"

# 边界用例：p_hat=1.0 时 se=0，Wald 区间塌缩为零宽度（这正是它的缺陷）
lo, hi = wald_ci(1.0, 50)
assert lo == 1.0 and hi == 1.0, "p_hat=1.0 时区间应为 (1.0, 1.0)"

print("✅ 练习 1 通过")

## ✏️ 练习 2：bootstrap 均值置信区间

很多 eval 指标没有现成的解析 CI 公式（平均得分、win rate、加权指标……）。通用武器是 <b>bootstrap</b>（模块 02 详讲）：**有放回重采样 $B$ 次，每次重算一遍指标，取经验分位数作为区间**。

实现 `bootstrap_mean_ci(x, B=2000, alpha=0.05)`：

1. 函数内固定 `rng = np.random.default_rng(0)`（保证可复现——自测依赖这一点，骨架已写好，勿改）；
2. 重复 $B$ 次：从 `x` 中**有放回**抽 `len(x)` 个样本，记录这份重采样的均值；
3. 返回这 $B$ 个均值的 $(\alpha/2,\; 1-\alpha/2)$ 经验分位数，tuple `(lo, hi)`。

**提示（向量化，免 for 循环）**：`idx = rng.integers(0, n, size=(B, n))` 一次生成全部重采样下标，`boot_means = x[idx].mean(axis=1)` 得到 $B$ 个均值；分位数用 `np.quantile(boot_means, q)`。用 for 循环逐次 `rng.integers(0, n, size=n)` 也完全可以。

In [ ]:
def bootstrap_mean_ci(x, B=2000, alpha=0.05):
    rng = np.random.default_rng(0)   # 固定种子：自测依赖可复现性，勿改
    x = np.asarray(x, dtype=float)
    n = len(x)
    # TODO: 生成 B 份有放回重采样的下标 idx（提示：rng.integers(0, n, size=(B, n))）
    # TODO: 计算 B 个 bootstrap 均值 boot_means
    # TODO: 返回 (np.quantile(boot_means, alpha / 2), np.quantile(boot_means, 1 - alpha / 2))
    raise NotImplementedError("完成上面三个 TODO 后删除此行")

In [ ]:
# ── 练习 2 自测（全部通过会打印 ✅）──────────────────────────
x = np.random.default_rng(42).normal(loc=2.0, scale=1.0, size=200)   # 真实均值 = 2.0

lo, hi = bootstrap_mean_ci(x)
assert lo < hi, "区间应满足 lo < hi"
assert lo < 2.0 < hi, f"95% CI 应盖住真实均值 2.0，你返回了 ({lo:.4f}, {hi:.4f})"
assert lo < x.mean() < hi, "CI 应包含样本均值"
assert 0.1 < (hi - lo) < 0.5, \
    f"n=200, sd=1 时区间宽度应在 0.1~0.5 之间（理论约 2*1.96/sqrt(200)≈0.28），你的宽度 {hi - lo:.4f}"

# 可复现性：函数内部固定了 rng(0)，两次调用必须完全一致
lo2, hi2 = bootstrap_mean_ci(x)
assert lo2 == lo and hi2 == hi, "两次调用结果不一致——确认 rng 是在函数内部以种子 0 创建的"

print("✅ 练习 2 通过")

## 📖 参考答案

**先自己做，再对照。** 下方 cell 是两题的参考实现：运行它会覆盖你的同名函数，可以回到上面的两个自测 cell 重跑验证。

两个自测 cell 都打印 ✅ 后，你的环境与统计手感都已就绪 ——
前往 **[01 · 评测分类学与 benchmark 全景](../01_benchmark_landscape/01_讲解.html)**。

In [ ]:
# ===== 📖 参考答案（先自己做，再对照）=========================

# --- 练习 1：Wald CI ---
def wald_ci(p_hat, n, z=1.96):
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    return (p_hat - z * se, p_hat + z * se)

# --- 练习 2：bootstrap 均值 CI ---
def bootstrap_mean_ci(x, B=2000, alpha=0.05):
    rng = np.random.default_rng(0)
    x = np.asarray(x, dtype=float)
    n = len(x)
    idx = rng.integers(0, n, size=(B, n))    # B 份有放回重采样的下标
    boot_means = x[idx].mean(axis=1)         # 每份重采样的均值
    lo = float(np.quantile(boot_means, alpha / 2))
    hi = float(np.quantile(boot_means, 1 - alpha / 2))
    return (lo, hi)

print("参考实现已加载，可回到上面的自测 cell 重跑验证。")
print("示例：wald_ci(0.8, 100) =", wald_ci(0.8, 100))